# Exercise 0: How Does an LLM API Call Work?

Prerequisite: Setup from README completed.

In [ ]:
import ollama

models = [m.model for m in ollama.list().models]
assert any("qwen3.5" in m for m in models), f"qwen3.5:4b not found: {models}"
print("OK")

## Part 1: A Single LLM Call

An LLM generates text, token by token. Everything we see in APIs like `ollama.chat()` (fields like `thinking`, `content`, `tool_calls`) is parsed from this text stream after the fact.

First, let's look at the raw output directly via the REST API.

In [ ]:
# Raw LLM output directly via the REST API
# raw=true: no parsing by Ollama, we see the text as it is generated
# Prompt in the Qwen3.5 chat template with <think> pre-fill (activates think mode)
import requests, json

OLLAMA_URL = str(ollama.Client()._client.base_url).rstrip("/")

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "<|im_start|>user\nWhat is 2 + 2?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "raw": True,
    "stream": False,
})

# <think> was pre-filled, so prepend it for the complete picture
raw_output = "<think>\n" + resp.json()["response"]
# print(repr(raw_output))
print(raw_output)

In [ ]:
# Without chat template: the model doesn't know where the answer should stop
# We abort after 3 seconds
import time

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "What is 2 + 2?",
    "raw": True,
    "stream": True,
}, stream=True)

tokens = []
start = time.time()
for line in resp.iter_lines():
    if time.time() - start > 3:
        break
    tokens.append(json.loads(line)["response"])

# print(repr("".join(tokens)))
print("".join(tokens))

In [ ]:
# ollama.chat() does the same thing, but:
# 1. Builds the prompt with special tokens automatically (RENDERER)
# 2. Parses <think>...</think> from the output into the "thinking" field (PARSER)
# 3. The rest becomes "content"
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
)

print(json.dumps(response.message.model_dump(), indent=2, default=str))

## Part 2: Statelessness

The chat API is stateless. The model only sees the `messages` we send in each call.

In [ ]:
# Without conversation history: two separate calls
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "What is my name?"}])
print("Call 2:", r2.message.content)

In [ ]:
# With conversation history: same two calls, but we pass the history along
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[
    {"role": "user", "content": "My name is Max."},
    {"role": "assistant", "content": r1.message.content},
    {"role": "user", "content": "What is my name?"},
])
print("Call 2:", r2.message.content)

## Part 3: Tool Calling

An LLM always only generates text. Tool calling is not a special feature, but a **convention for structured output**: the model was trained to output JSON instead of free text for certain prompts.

We can first build this by hand, using only a system prompt.

In [ ]:
# "Tool calling" without the tools= parameter, using only a prompt
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": 'When you need to calculate, respond ONLY with this JSON:\n{"tool": "calculator", "expression": "<expression>"}\nNo other text.'},
        {"role": "user", "content": "Calculate 17 * 23"},
    ],
)

print("=== Raw Output (just text) ===")
print(response.message.content)

In [ ]:
# We can parse the text output ourselves and execute the tool
import json as _json

parsed = _json.loads(response.message.content)
print("Parsed:", parsed)

# This is tool calling: the model outputs structured text, we parse and execute.
print(parsed["expression"])
result = eval(parsed["expression"])
print("Result:", result)

This works, but it's fragile: we have to formulate the prompt precisely, define the JSON format ourselves, and parse the output manually. The `tools=` parameter automates exactly this:
- It injects the tool definitions into the prompt (in the format the model saw during training)
- It parses the output and delivers structured `tool_calls` instead of raw text

From here on we use `tools=`. The tool definition describes to the model which functions are available. The tool implementation is the Python function we execute.

In [ ]:
# Tool definition: describes to the model which function is available
# Reference: https://github.com/ollama/ollama/blob/main/docs/api.md#chat-request-with-tools
calculator_tool = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression"}
                },
                "required": ["expression"],
            },
        },
    }
]

In [ ]:
# Call the LLM with the tool definition, inspect the full response
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Calculate 17 * 23"}],
    tools=calculator_tool,
)

print(json.dumps(response.model_dump(), indent=2, default=str))

In the response we see:
- `content` is empty: the model did not return any text
- `tool_calls` is populated: the model wants to call `calculator` with `{"expression": "17 * 23"}`

The model did not calculate anything. It returned **which function** it wants to call with **which arguments**. Now we need the Python function that actually executes it.

In [ ]:
# Tool implementation: the Python function we execute
import numexpr, math

def calculator(expression: str) -> str:
    result = numexpr.evaluate(expression, local_dict={"pi": math.pi, "e": math.e})
    return f"Result: {float(result)}"

# Test: call directly
print(calculator("17 * 23"))

In [ ]:
# Connection: how do we find the right function for a tool call?
# The name in the tool call ("calculator") must match the key in tool_map.

tool_map = {"calculator": calculator}

tc = response.message.tool_calls[0]       # Tool call from the response
print(f"name:      {tc.function.name}")    # "calculator"
print(f"arguments: {tc.function.arguments}")  # {"expression": "17 * 23"}

func = tool_map[tc.function.name]          # Look up the calculator function
result = func(**tc.function.arguments)     # calculator(expression="17 * 23")
print(f"result:    {result}")

## Part 4: Tool-Use Loop

We have executed the tool and got a result. But the API is stateless: the model knows nothing about the execution. We must include the result as a `{"role": "tool"}` message in the history and call the model again.

In [ ]:
# Complete tool-use loop
messages = [{"role": "user", "content": "Calculate 17 * 23"}]

# 1. Call the LLM
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Check: does the model want to call a tool?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Execute the tool
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    result = func(**tc.function.arguments)  # why ** — explain

    # 4. Send the tool result back to the model
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages to LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Final Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

In [ ]:
# Sanity check: what happens when the model decides AGAINST using a tool?
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Calculate 17 * 23. Answer directly, do NOT use a tool."}],
    tools=calculator_tool,
)

print("tool_calls:", response.message.tool_calls)
print("content:   ", json.dumps(response.model_dump(), indent=2, default=str))


# add if block

## Summary

1. `ollama.chat()` takes a `messages` list and returns a `ChatResponse` object.
2. The API is stateless. The model only sees the messages we send in each call.
3. With `tools=`, the model can return tool calls. We check for this with `if response.message.tool_calls`.
4. The tool definition (JSON object with name, description, parameter schema) tells the model what is available. The tool implementation (Python function) is executed by us. They are connected via `tool_map`.
5. The tool-use loop (call LLM → check → execute tool → send result back → call LLM again) is what an agent automates.

→ **Exercise 1**: implement this loop.